# Illumination-Correct and Export 16-bit OME-TIFFs

This notebook loads microscopy images, applies **illumination correction**, and saves the
corrected images as **16-bit OME-TIFFs with no intensity rescaling**, using the aicsimageio
OME-TIFF writer. Output files are renamed using the plate layout metadata.

## What this notebook does
1. Loads a plate layout CSV with experimental conditions
2. Matches it to the image metadata
3. Selects a subset of images (use `additional_filter` for this)
4. Applies illumination correction
5. Saves each corrected image as a 16-bit OME-TIFF, named from the plate layout
6. Optionally saves the matching segmentation as a transparent PNG with white
   outlines (for overlaying on the final image)

## Before you start
- Prepare your plate layout CSV
- Know the path to your images, the illumination correction file, and (optionally) segmentations
- Decide how to narrow the image set with `additional_filter`


## Import Libraries

In [79]:
# Standard library
import os
import glob
import warnings
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Image IO (aicsimageio)
from aicsimageio import AICSImage, readers
from aicsimageio.writers import OmeTiffWriter

# Segmentation outlines
import mahotas as mh
from PIL import Image

# Progress bars
from tqdm.auto import tqdm

# Illumination correction (core step for this notebook)
from blimp.preprocessing.illumination_correction import IlluminationCorrection

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


---
# Configuration Section
**Edit the parameters below to match your experiment**

In [80]:
# ============================================================================
# FILE PATHS
# ============================================================================

# Path to your plate layout CSV file (contains experimental metadata)
plate_layout_file = "/srv/scratch/z3532965/src/blana/Scott/20260626_POLR2A_heterogeneity/METADATA/20260626_single_cell_POLR2A_plate_layout.csv"

# Path to directory containing images (and their *.csv metadata)
image_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/OME-TIFF-MIP/"

# Path to illumination correction file (required for this notebook)
illumination_correction_file = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/illumination_correction.pkl"

# Output directory for the corrected 16-bit OME-TIFFs
output_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/CORRECTED_TIFF"

# Regex to extract the well name from the image filename (optional)
# Example: for "WellA01_Channel..." use r'Well([A-Z]\d{2})_Channel'
# Set to None to use the well column from the image metadata as-is
well_extraction_pattern = r'Well([A-Z]\d{2})_Channel'


In [81]:
# Load plate layout
plate_layout = None
if os.path.exists(plate_layout_file):
    plate_layout = pd.read_csv(plate_layout_file, dtype=str)
    plate_layout = plate_layout.where(pd.notnull(plate_layout), "None")
    print(f"✓ Plate layout: {len(plate_layout)} rows\n")

    for col in plate_layout.columns:
        unique_values = plate_layout[col].unique()
        n_unique = len(unique_values)
        if n_unique <= 8 and all(len(str(v)) <= 20 for v in unique_values):
            print(f"  {col}: {list(unique_values)}")
        else:
            print(f"  {col}: {n_unique} unique values")

    print(f"\n{plate_layout.head()}\n")
else:
    print(f"⚠ Plate layout file not found: {plate_layout_file}\n")

# Load image metadata
image_metadata = None
if os.path.exists(image_dir):
    image_metadata_files = glob.glob(os.path.join(image_dir, "*.csv"))
    if image_metadata_files:
        image_metadata = pd.concat((pd.read_csv(f, dtype=str) for f in image_metadata_files), ignore_index=True)
        print(f"✓ Image metadata: {len(image_metadata)} rows from {len(image_metadata_files)} file(s)\n")

        for col in image_metadata.columns:
            unique_values = image_metadata[col].unique()
            n_unique = len(unique_values)
            if n_unique <= 8 and all(len(str(v)) <= 20 for v in unique_values):
                print(f"  {col}: {list(unique_values)}")
            else:
                print(f"  {col}: {n_unique} unique values")

        print(f"\n{image_metadata.head()}")
    else:
        print(f"⚠ No CSV files found in: {image_dir}")
else:
    print(f"⚠ Image directory not found: {image_dir}")


✓ Plate layout: 384 rows

  well: 384 unique values
  CELLS: ['mAC-POLR2A', 'mCherry-POLR2A', 'OsTIR(F74G)']
  PRIMARY_RAT: ['0', '3E10', '3E8', 'None']
  PRIMARY_RABBIT: ['0', 'None', 'D8L4Y']
  SECONDARY_488: ['0', 'None', 'Rabbit', 'Rat']
  SECONDARY_568: ['0', 'Rabbit', 'None', 'Rat']
  SECONDARY_647: ['0', 'Rat', 'None', 'Rabbit']
  DAPI: ['0', 'DAPI', 'None']

  well       CELLS PRIMARY_RAT PRIMARY_RABBIT SECONDARY_488 SECONDARY_568  \
0  A01  mAC-POLR2A           0              0             0             0   
1  A02  mAC-POLR2A           0              0             0             0   
2  A03  mAC-POLR2A           0              0             0             0   
3  A04  mAC-POLR2A           0              0             0             0   
4  A05  mAC-POLR2A           0              0             0             0   

  SECONDARY_647 DAPI  
0             0    0  
1             0    0  
2             0    0  
3             0    0  
4             0    0  

✓ Image metadata: 1296 rows f

In [82]:
# ============================================================================
# COLUMN MAPPING
# ============================================================================

# Column in the plate layout CSV that contains well identifiers
plate_layout_well_column = "well"

# Column in the image metadata CSV that contains well identifiers
image_metadata_well_column = "well"

# Column in the image metadata CSV that contains the image filename
image_metadata_filename_column = "filename_ome_tiff"

# Column in the image metadata that contains the field ID
image_metadata_field_column = "field_id"


In [83]:
# ============================================================================
# IMAGE SELECTION
# ============================================================================

# Keep only a single field ID (string), or set to None to keep all fields.
selected_field_id = '6'

# additional_filter is the MAIN mechanism for selecting a smaller set of images.
# It is a pandas .query() string evaluated against the merged plate-layout columns.
# Set to None for no additional filtering.
additional_filter = (
    "PRIMARY_RAT == '3E10' "
    "and PRIMARY_RABBIT == 'D8L4Y' "
    "and SECONDARY_488 == 'None' "
    "and SECONDARY_568 == 'Rat' "
    "and SECONDARY_647 == 'Rabbit'"
)


In [84]:
# ============================================================================
# NAMING
# ============================================================================

# Plate-layout columns used to build the output filename.
# The output name is:  <grouping_variables joined by _>_<well>_Field<field>.ome.tiff
grouping_variables = ["CELLS", "PRIMARY_RAT", "PRIMARY_RABBIT", "SECONDARY_488", "SECONDARY_568", "SECONDARY_647"]

# If True, keep only one representative well per unique combination of grouping_variables.
# If False, export every image that passes the filters.
select_one_well_per_condition = False


In [85]:
# ============================================================================
# OUTPUT SETTINGS
# ============================================================================

# Channels (0-based indices) to include in the output TIFF, or None for ALL channels.
# No rescaling or colormap is applied - raw corrected intensities are written.
channels_to_save = [1, 2, 4, 5]

# Suffix / extension for output files
output_suffix = ".ome.tiff"


In [86]:
# ============================================================================
# SEGMENTATION OUTLINE (optional)
# ============================================================================

# Directory containing segmentation label images. Set to None to skip outlines.
# Segmentation filenames are expected to match the image filenames exactly.
segmentation_dir = "/srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/SEGMENTATION/"

# Channel (0-based) in the segmentation image that holds the object labels
segmentation_channel = 0

# Thin single-pixel outlines (True) or thicker dilated outlines (False)
segmentation_thin_outlines = True


---
# Code Section
**You shouldn't need to edit anything below this line**

## Validate Configuration

In [87]:
errors = []

if plate_layout is None:
    errors.append(f"Plate layout file not found: {plate_layout_file}")
if image_metadata is None:
    errors.append(f"No image metadata found in: {image_dir}")
if not os.path.exists(image_dir):
    errors.append(f"Image directory not found: {image_dir}")

if illumination_correction_file is None:
    errors.append("No illumination correction file specified (this notebook requires one)")
elif not os.path.exists(illumination_correction_file):
    errors.append(f"Illumination correction file not found: {illumination_correction_file}")

if segmentation_dir is not None and not os.path.exists(segmentation_dir):
    errors.append(f"Segmentation directory not found: {segmentation_dir}")

if errors:
    print("❌ Configuration errors found:")
    for e in errors:
        print(f"  • {e}")
    raise ValueError("Please fix configuration errors above")
else:
    print("✓ Configuration validated successfully")
    print("\nSettings summary:")
    print(f"  • Plate layout: {len(plate_layout)} wells loaded")
    print(f"  • Image metadata: {len(image_metadata)} images loaded")
    print(f"  • Field ID: {selected_field_id if selected_field_id is not None else 'ALL'}")
    print(f"  • Channels: {channels_to_save if channels_to_save is not None else 'ALL'}")
    print(f"  • Naming by: {grouping_variables}")
    print(f"  • Segmentation outlines: {'Yes' if segmentation_dir is not None else 'No'}")

# Extract well name from filename if a pattern is provided
if well_extraction_pattern is not None:
    if image_metadata_filename_column in image_metadata.columns:
        image_metadata['well_name_extracted'] = image_metadata[image_metadata_filename_column].str.extract(well_extraction_pattern)
        print(f"\n✓ Extracted well names using pattern: {well_extraction_pattern}")
        print(f"  Example: {image_metadata[image_metadata_filename_column].iloc[0]} → {image_metadata['well_name_extracted'].iloc[0]}")
    else:
        raise ValueError(f"Cannot extract well names: column '{image_metadata_filename_column}' not found in image metadata")


✓ Configuration validated successfully

Settings summary:
  • Plate layout: 384 wells loaded
  • Image metadata: 1296 images loaded
  • Field ID: 6
  • Channels: [1, 2, 4, 5]
  • Naming by: ['CELLS', 'PRIMARY_RAT', 'PRIMARY_RABBIT', 'SECONDARY_488', 'SECONDARY_568', 'SECONDARY_647']
  • Segmentation outlines: Yes

✓ Extracted well names using pattern: Well([A-Z]\d{2})_Channel
  Example: WellK08_ChannelNone,488,561,XXX,647,405_Seq0101_0001.ome.tiff → K08


## Load Illumination Correction

In [88]:
print(f"Loading illumination correction from: {illumination_correction_file}")
illumination_correction = IlluminationCorrection(from_file=illumination_correction_file)
print("✓ Illumination correction loaded")


Loading illumination correction from: /srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/illumination_correction.pkl
✓ Illumination correction loaded


## Validate and Merge Metadata

Validate column names and merge the plate layout with the image metadata.

In [89]:
# Plate layout checks
if plate_layout_well_column not in plate_layout.columns:
    raise ValueError(f"Column '{plate_layout_well_column}' not found in plate layout. "
                     f"Available columns: {list(plate_layout.columns)}")

missing_vars = [var for var in grouping_variables if var not in plate_layout.columns]
if missing_vars:
    raise ValueError(f"Grouping variables not found in plate layout: {missing_vars}")

# Image metadata checks
required_cols = [image_metadata_filename_column, image_metadata_field_column]
if well_extraction_pattern is not None:
    if 'well_name_extracted' not in image_metadata.columns:
        raise ValueError("Well extraction pattern provided but well_name_extracted column not found")
    merge_column = 'well_name_extracted'
else:
    required_cols.append(image_metadata_well_column)
    merge_column = image_metadata_well_column

missing_cols = [col for col in required_cols if col not in image_metadata.columns]
if missing_cols:
    raise ValueError(f"Required columns not found in image metadata: {missing_cols}. "
                     f"Available columns: {list(image_metadata.columns)}")

# Merge
print("Merging plate layout with image metadata...")
image_metadata_annotated = image_metadata.merge(
    plate_layout,
    left_on=merge_column,
    right_on=plate_layout_well_column,
    how='inner'
)
print(f"✓ Merged data: {len(image_metadata_annotated)} images with annotations")

if len(image_metadata_annotated) == 0:
    raise ValueError("No matches found between plate layout and image metadata. "
                     f"Merging on: image_metadata['{merge_column}'] = plate_layout['{plate_layout_well_column}']")


Merging plate layout with image metadata...
✓ Merged data: 1296 images with annotations


## Filter Images

Select images by field ID (optional) and `additional_filter` (main selection).

In [90]:
filtered_images = image_metadata_annotated.copy()

# Optional field filter
if selected_field_id is not None:
    print(f"Filtering for field ID: {selected_field_id}")
    filtered_images = filtered_images[
        filtered_images[image_metadata_field_column] == selected_field_id
    ].copy()
    print(f"✓ After field filter: {len(filtered_images)} images")

# Main selection: additional_filter
if additional_filter is not None:
    print(f"Applying additional filter: {additional_filter}")
    filtered_images = filtered_images.query(additional_filter)
    print(f"✓ After additional filter: {len(filtered_images)} images")

if len(filtered_images) == 0:
    raise ValueError("No images remaining after filtering. Check your filter criteria.")

# Optionally keep one representative well per condition
if select_one_well_per_condition:
    filtered_images = filtered_images.groupby(grouping_variables, as_index=False).first()
    print(f"✓ Selected {len(filtered_images)} representative wells (one per condition)")

print("\nImages to be processed per condition:")
print(filtered_images.groupby(grouping_variables).size())


Filtering for field ID: 6
✓ After field filter: 144 images
Applying additional filter: PRIMARY_RAT == '3E10' and PRIMARY_RABBIT == 'D8L4Y' and SECONDARY_488 == 'None' and SECONDARY_568 == 'Rat' and SECONDARY_647 == 'Rabbit'
✓ After additional filter: 13 images

Images to be processed per condition:
CELLS        PRIMARY_RAT  PRIMARY_RABBIT  SECONDARY_488  SECONDARY_568  SECONDARY_647
OsTIR(F74G)  3E10         D8L4Y           None           Rat            Rabbit            1
mAC-POLR2A   3E10         D8L4Y           None           Rat            Rabbit           12
dtype: int64


## Prepare Output Filenames

Build output filenames from the plate-layout grouping variables plus well and field.

In [91]:
print("Generating output filenames...")

# Condition id from grouping variables (from the plate layout)
if len(grouping_variables) == 1:
    filtered_images['condition_id'] = filtered_images[grouping_variables[0]].astype(str)
else:
    filtered_images['condition_id'] = filtered_images[grouping_variables[0]].astype(str)
    for var in grouping_variables[1:]:
        filtered_images['condition_id'] = filtered_images['condition_id'] + '_' + filtered_images[var].astype(str)

# Add well and field
filtered_images['base_filename'] = (
    filtered_images['condition_id'] + '_' +
    filtered_images[image_metadata_well_column].astype(str) + '_' +
    'Field' + filtered_images[image_metadata_field_column].astype(str)
)

# Make filesystem-safe (replace spaces and path separators)
filtered_images['base_filename'] = filtered_images['base_filename'].str.replace(r'[\\/ ]', '-', regex=True)

print(f"✓ Generated filenames for {len(filtered_images)} images")
print("\nExample filenames:")
print(filtered_images['base_filename'].head(3).tolist())


Generating output filenames...
✓ Generated filenames for 13 images

Example filenames:
['mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G09_Field6', 'mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_H09_Field6', 'mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G06_Field6']


## Helper: Segmentation Outline PNG

Builds a transparent RGBA PNG with white object outlines, matching the image
dimensions so it can be overlaid on the final image.

In [92]:
def make_outline_png(label_image, thin=True):
    """White object outlines on a transparent background (RGBA PIL image)."""
    label_image = np.asarray(label_image)

    # Thin single-pixel borders; optionally dilate for thicker outlines
    borders = mh.labeled.borders(label_image)
    if not thin:
        borders = mh.morph.dilate(borders)

    h, w = borders.shape
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[borders, :3] = 255   # white
    rgba[borders, 3] = 255    # opaque only where there is a border
    return Image.fromarray(rgba, mode="RGBA")

print("✓ Outline helper defined")


✓ Outline helper defined


## Illumination-Correct and Save 16-bit OME-TIFFs

Each image is loaded, illumination-corrected, and written as a 16-bit OME-TIFF with the
original channel names and pixel sizes preserved. No intensity rescaling is applied - the
corrected data is cast back to the source integer dtype (rounded and clipped only to avoid
overflow).

In [93]:
output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)
print(f"✓ Output directory: {output_path}")
print(f"Processing {len(filtered_images)} images...\n")

processing_errors = []
saved = 0
outlines_saved = 0

for idx, row in tqdm(filtered_images.iterrows(), total=len(filtered_images), desc="Saving corrected TIFFs"):
    image_path = Path(image_dir) / row[image_metadata_filename_column]
    base_filename = row['base_filename']

    try:
        # Load
        aics_image = AICSImage(image_path, reader=readers.ome_tiff_reader.OmeTiffReader)
        original_dtype = np.dtype(aics_image.dtype)

        # Carry metadata into the output
        channel_names = list(aics_image.channel_names)
        physical_pixel_sizes = aics_image.physical_pixel_sizes

        # Illumination correction
        corrected = illumination_correction.correct(aics_image)

        # Full 5D data in TCZYX order
        data = corrected.get_image_data("TCZYX")

        # Optional channel subset (no rescaling)
        if channels_to_save is not None:
            data = data[:, channels_to_save, :, :, :]
            channel_names = [channel_names[c] for c in channels_to_save]

        # Preserve 16-bit output without rescaling:
        # if correction produced floats, round + clip to the source integer range and cast back.
        if not np.issubdtype(data.dtype, np.integer):
            info = np.iinfo(original_dtype) if np.issubdtype(original_dtype, np.integer) else np.iinfo(np.uint16)
            data = np.clip(np.rint(data), info.min, info.max).astype(info.dtype)
        elif data.dtype != original_dtype:
            data = data.astype(original_dtype)

        # Save as OME-TIFF using aicsimageio's writer
        output_file = output_path / f"{base_filename}{output_suffix}"
        OmeTiffWriter.save(
            data,
            str(output_file),
            dim_order="TCZYX",
            channel_names=channel_names,
            physical_pixel_sizes=physical_pixel_sizes,
        )
        saved += 1

        # Matching segmentation outline PNG (white outlines, transparent background)
        if segmentation_dir is not None:
            seg_file_path = Path(segmentation_dir) / row[image_metadata_filename_column]
            if seg_file_path.exists():
                seg_image = AICSImage(seg_file_path, reader=readers.ome_tiff_reader.OmeTiffReader)
                seg_array = seg_image.get_image_data('YX', Z=0, C=segmentation_channel, T=0)
                outline = make_outline_png(seg_array, thin=segmentation_thin_outlines)
                outline_file = output_path / f"{base_filename}_segmentation_outline.png"
                outline.save(str(outline_file))
                outlines_saved += 1
            else:
                warnings.warn(f"Segmentation file not found: {seg_file_path}")

    except Exception as e:
        msg = f"Error processing {image_path}: {e}"
        processing_errors.append(msg)
        warnings.warn(msg)

print(f"\n✓ Done. Saved {saved} / {len(filtered_images)} images.")
if segmentation_dir is not None:
    print(f"  • Segmentation outline PNGs saved: {outlines_saved}")
if processing_errors:
    print(f"  • Errors encountered: {len(processing_errors)}")
    for m in processing_errors[:5]:
        print("   -", m)
    if len(processing_errors) > 5:
        print(f"   ... and {len(processing_errors) - 5} more")


✓ Output directory: /srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/CORRECTED_TIFF
Processing 13 images...



Saving corrected TIFFs: 100%|██████████| 13/13 [00:32<00:00,  2.50s/it]


✓ Done. Saved 13 / 13 images.
  • Segmentation outline PNGs saved: 13


## Summary

In [94]:
print("=" * 70)
print("PROCESSING COMPLETE")
print("=" * 70)
print(f"\nOutput directory: {output_path}")

tiffs = sorted(output_path.glob(f"*{output_suffix}"))
print(f"\n16-bit OME-TIFFs written: {len(tiffs)}")
for f in tiffs[:10]:
    print("  •", f.name)
if len(tiffs) > 10:
    print(f"  ... and {len(tiffs) - 10} more")

outlines = sorted(output_path.glob("*_segmentation_outline.png"))
print(f"\nSegmentation outline PNGs written: {len(outlines)}")

print("\nConditions processed:")
for condition, count in filtered_images.groupby(grouping_variables).size().items():
    print(f"  • {condition}: {count} image(s)")


PROCESSING COMPLETE

Output directory: /srv/scratch/berrylab/z3532965/systems_Ti2/20260626_POLR2A_heterogeneity/20260629_134305_894/CORRECTED_TIFF

16-bit OME-TIFFs written: 26
  • OsTIR(F74G)_3E10_D8L4Y_None_Rat_Rabbit_M05_Field6.ome.tiff
  • OsTIR(F74G)_3E10_D8L4Y_Rat_None_Rabbit_M07_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G04_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G05_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G06_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G07_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G08_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_G09_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_H04_Field6.ome.tiff
  • mAC-POLR2A_3E10_D8L4Y_None_Rat_Rabbit_H05_Field6.ome.tiff
  ... and 16 more

Segmentation outline PNGs written: 26

Conditions processed:
  • ('OsTIR(F74G)', '3E10', 'D8L4Y', 'None', 'Rat', 'Rabbit'): 1 image(s)
  • ('mAC-POLR2A', '3E10', 'D8L4Y', 'None', 'Ra